# 05 — Feature Engineering

## Objective

This notebook engineers new, domain-informed features on top of the cleaned dataset from Notebook 02, using the exact same patient-level split established in Notebook 03 (`patient_split.pkl`).

### Why Feature Engineering Matters Here

The baseline pipeline (Notebook 03) used the 44 raw features with minimal transformation: numerical scaling and one-hot encoding of categoricals. This produced 2,304 sparse, mostly uninformative dummy columns — largely driven by high-cardinality features like `diag_1`, `diag_2`, `diag_3` (700+ raw ICD-9 codes each).

This notebook focuses on:
1. **Clinical grouping of ICD-9 diagnosis codes** into a small number of clinically meaningful categories (following the approach used in the original Strack et al. 2014 study).
2. **Derived utilization features** — combining existing counts into ratios and totals that may carry more signal than their raw components.
3. **Medication change indicators** — summarizing the 23 individual drug columns into higher-level signals.

### Principles (carried over from Notebook 03)

- `random_state = 42`
- No feature is engineered using validation or test data — every derived statistic is computed from training data only, or is a pure row-wise transformation with no data leakage risk.
- The test set is not touched in this notebook.
- Every engineered feature is evaluated against the baseline using the same two champion models (CatBoost, LightGBM) plus the XGBoost reference model, on the validation set.

In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_data.csv"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

# Load the cleaned dataset (output of Notebook 02)
df = pd.read_csv(DATA_PATH, low_memory=False)
print("Cleaned dataset shape:", df.shape)

# Reapply the expired/hospice filter (same decision as Notebook 03)
EXPIRED_HOSPICE_CODES = [11, 13, 14, 19, 20, 21]
expired_hospice_mask = df["discharge_disposition_id"].isin(EXPIRED_HOSPICE_CODES)
df = df.loc[~expired_hospice_mask].reset_index(drop=True)
print("Shape after removing expired/hospice encounters:", df.shape)

# Convert target to binary (same mapping as Notebook 03)
df["readmitted"] = df["readmitted"].map({"NO": 0, "<30": 1, ">30": 1})

# Reload the exact patient-level split from Notebook 03
split_patients = joblib.load(MODELS_DIR / "patient_split.pkl")
train_patients = split_patients["train_patients"]
val_patients = split_patients["val_patients"]
test_patients = split_patients["test_patients"]

train_df = df[df["patient_nbr"].isin(train_patients)].copy()
val_df = df[df["patient_nbr"].isin(val_patients)].copy()
test_df = df[df["patient_nbr"].isin(test_patients)].copy()

print("\nReconstructed split shapes:")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Cleaned dataset shape: (101766, 47)
Shape after removing expired/hospice encounters: (99343, 47)

Reconstructed split shapes:
Train: (63444, 47)
Validation: (15775, 47)
Test: (20124, 47)


## Clinical Grouping of Diagnosis Codes (ICD-9)

`diag_1`, `diag_2`, and `diag_3` each contain 700–800 distinct raw ICD-9 codes. One-hot encoding these directly (as done in the baseline) produces thousands of sparse, mostly single-occurrence dummy columns — this is the single largest contributor to the 2,304-dimensional baseline feature space.

Following the grouping scheme used in the original study on this dataset (Strack et al., 2014, *Impact of HbA1c Measurement on Hospital Readmission Rates*), each ICD-9 code is mapped into one of a small number of clinically meaningful diagnostic categories:

- Circulatory
- Respiratory
- Digestive
- Diabetes
- Injury
- Musculoskeletal
- Genitourinary
- Neoplasms
- Other

This reduces diagnosis-related dimensionality from ~2,000+ one-hot columns down to a handful of interpretable categories per diagnosis slot, while preserving the clinical signal the codes carry.

The mapping is defined using ICD-9 code ranges and applied identically to `diag_1`, `diag_2`, and `diag_3`. Codes are strings that may include decimals (e.g., `"250.83"`) or the letter prefixes `V` and `E` (external causes / supplemental classification), which are handled explicitly.

In [2]:
def map_icd9_to_category(code):
    """
    Map a raw ICD-9 diagnosis code string to a clinical category,
    following the grouping used in Strack et al. (2014).
    """
    if pd.isna(code):
        return "Missing"

    code = str(code)

    # Supplemental classification codes
    if code.startswith("V"):
        return "Other"
    if code.startswith("E"):
        return "Other"

    try:
        numeric_code = float(code)
    except ValueError:
        return "Other"

    # Diabetes (ICD-9 250.xx)
    if 250 <= numeric_code < 251:
        return "Diabetes"

    # Circulatory (390-459, 785)
    if (390 <= numeric_code <= 459) or numeric_code == 785:
        return "Circulatory"

    # Respiratory (460-519, 786)
    if (460 <= numeric_code <= 519) or numeric_code == 786:
        return "Respiratory"

    # Digestive (520-579, 787)
    if (520 <= numeric_code <= 579) or numeric_code == 787:
        return "Digestive"

    # Injury and poisoning (800-999)
    if 800 <= numeric_code <= 999:
        return "Injury"

    # Musculoskeletal (710-739)
    if 710 <= numeric_code <= 739:
        return "Musculoskeletal"

    # Genitourinary (580-629, 788)
    if (580 <= numeric_code <= 629) or numeric_code == 788:
        return "Genitourinary"

    # Neoplasms (140-239)
    if 140 <= numeric_code <= 239:
        return "Neoplasms"

    return "Other"


diagnosis_columns = ["diag_1", "diag_2", "diag_3"]

for df_split, name in [(train_df, "Train"), (val_df, "Validation"), (test_df, "Test")]:
    for col in diagnosis_columns:
        df_split[f"{col}_category"] = df_split[col].apply(map_icd9_to_category)

print("New diagnosis category columns created: diag_1_category, diag_2_category, diag_3_category")

print("\nCategory distribution in diag_1 (Train):")
print(train_df["diag_1_category"].value_counts())

print("\nCategory distribution in diag_2 (Train):")
print(train_df["diag_2_category"].value_counts())

print("\nCategory distribution in diag_3 (Train):")
print(train_df["diag_3_category"].value_counts())

New diagnosis category columns created: diag_1_category, diag_2_category, diag_3_category

Category distribution in diag_1 (Train):
diag_1_category
Circulatory        18959
Other              11344
Respiratory         8852
Digestive           5961
Diabetes            5584
Injury              4351
Musculoskeletal     3196
Genitourinary       3187
Neoplasms           1998
Missing               12
Name: count, dtype: int64

Category distribution in diag_2 (Train):
diag_2_category
Circulatory        19977
Other              16596
Diabetes            8108
Respiratory         6706
Genitourinary       5137
Digestive           2601
Injury              1520
Neoplasms           1454
Musculoskeletal     1139
Missing              206
Name: count, dtype: int64

Category distribution in diag_3 (Train):
diag_3_category
Circulatory        18778
Other              18257
Diabetes           10829
Respiratory         4498
Genitourinary       4123
Digestive           2480
Injury              1269
Musculosk

## Derived Utilization Features

The dataset includes several raw utilization counts: `number_outpatient`, `number_emergency`, `number_inpatient` (visits in the year prior to this encounter), and `number_diagnoses`, `num_procedures`, `num_medications`, `time_in_hospital` (within this encounter).

Two types of derived features are created:

1. **Total prior utilization**: `number_outpatient + number_emergency + number_inpatient` — patients with high total utilization in the prior year are clinically known to be at higher readmission risk; combining these three counts may capture this more directly than the individual components.

2. **Care intensity ratios**: features like `num_medications / time_in_hospital` and `num_lab_procedures / time_in_hospital` normalize treatment intensity by length of stay, which may separate "intensive short stay" from "prolonged low-intensity stay" patterns that raw counts alone conflate.

All of these are pure row-wise arithmetic transformations of existing columns — no information from other rows or other splits is used, so there is no leakage risk in computing them independently on train/val/test.

In [3]:
for df_split in [train_df, val_df, test_df]:
    # Total prior-year utilization
    df_split["total_prior_utilization"] = (
        df_split["number_outpatient"]
        + df_split["number_emergency"]
        + df_split["number_inpatient"]
    )

    # Care intensity ratios (avoid division by zero: time_in_hospital >= 1 always in this dataset)
    df_split["medications_per_day"] = (
        df_split["num_medications"] / df_split["time_in_hospital"]
    )
    df_split["lab_procedures_per_day"] = (
        df_split["num_lab_procedures"] / df_split["time_in_hospital"]
    )

    # Total procedures (lab + non-lab), a simple combined workload indicator
    df_split["total_procedures"] = (
        df_split["num_procedures"] + df_split["num_lab_procedures"]
    )

print("New utilization features created:")
print("- total_prior_utilization")
print("- medications_per_day")
print("- lab_procedures_per_day")
print("- total_procedures")

print("\nSummary statistics (Train):")
new_cols = ["total_prior_utilization", "medications_per_day", "lab_procedures_per_day", "total_procedures"]
print(train_df[new_cols].describe().round(3))

print("\nMissing values check:")
print(train_df[new_cols].isna().sum())

New utilization features created:
- total_prior_utilization
- medications_per_day
- lab_procedures_per_day
- total_procedures

Summary statistics (Train):
       total_prior_utilization  medications_per_day  lab_procedures_per_day  \
count                63444.000            63444.000               63444.000   
mean                     1.188                5.079                  14.307   
std                      2.238                3.821                  11.986   
min                      0.000                0.083                   0.071   
25%                      0.000                2.600                   6.460   
50%                      0.000                4.000                  10.857   
75%                      2.000                6.250                  18.200   
max                     42.000               42.000                  92.000   

       total_procedures  
count         63444.000  
mean             44.138  
std              19.777  
min               1.000  
25%

## Medication Change Indicators

The dataset contains 23 individual diabetes medication columns (`metformin`, `insulin`, `glipizide`, etc.), each with values like `No`, `Steady`, `Up`, `Down`. One-hot encoding all 23 individually (as in the baseline) produces many sparse columns, most dominated by the `No` category.

Two summary indicators are derived instead:

1. **`num_medications_changed`**: count of medications with a dosage change (`Up` or `Down`) during this encounter. A higher number of changes may reflect more unstable glycemic control, which is clinically associated with readmission risk.

2. **`num_medications_prescribed`**: count of medications that are not `No` (i.e., actively prescribed, regardless of whether the dose changed) — a proxy for regimen complexity.

These are computed from the 23 raw medication columns only (no leakage), and the original raw columns are still preserved for the encoding pipeline — these are additive features, not replacements.

In [4]:
medication_columns = [
    "metformin", "repaglinide", "nateglinide", "chlorpropamide", "glimepiride",
    "acetohexamide", "glipizide", "glyburide", "tolbutamide", "pioglitazone",
    "rosiglitazone", "acarbose", "miglitol", "troglitazone", "tolazamide",
    "insulin", "glyburide-metformin", "glipizide-metformin",
    "glimepiride-pioglitazone", "metformin-rosiglitazone", "metformin-pioglitazone",
]

for df_split in [train_df, val_df, test_df]:
    df_split["num_medications_changed"] = (
        df_split[medication_columns].isin(["Up", "Down"]).sum(axis=1)
    )
    df_split["num_medications_prescribed"] = (
        (df_split[medication_columns] != "No").sum(axis=1)
    )

print("New medication features created:")
print("- num_medications_changed")
print("- num_medications_prescribed")

print("\nSummary statistics (Train):")
print(train_df[["num_medications_changed", "num_medications_prescribed"]].describe().round(3))

print("\nDistribution of num_medications_changed (Train):")
print(train_df["num_medications_changed"].value_counts().sort_index())

New medication features created:
- num_medications_changed
- num_medications_prescribed

Summary statistics (Train):
       num_medications_changed  num_medications_prescribed
count                63444.000                   63444.000
mean                     0.287                       1.191
std                      0.487                       0.927
min                      0.000                       0.000
25%                      0.000                       1.000
50%                      0.000                       1.000
75%                      1.000                       2.000
max                      4.000                       6.000

Distribution of num_medications_changed (Train):
num_medications_changed
0    46180
1    16376
2      819
3       66
4        3
Name: count, dtype: int64


## Finalize Engineered Feature Set

Decision on raw vs. engineered columns for this stage:

- **`diag_1`, `diag_2`, `diag_3` (raw ICD-9 codes) are dropped.** They are superseded by `diag_1_category`, `diag_2_category`, `diag_3_category`, which capture the same clinical information at a much lower, more generalizable dimensionality (9 categories vs. 700+ raw codes).
- **The 21 individual medication columns are kept as-is.** Each has only 4 possible values (`No`, `Steady`, `Up`, `Down`), so their one-hot dimensionality is manageable, and dropping them would discard drug-specific information (e.g., insulin is known from the original study to be independently predictive) that the two new summary counts do not fully capture.
- All other baseline features (demographics, admission/discharge codes, lab results, `change`, `diabetesMed`) are kept unchanged.

This keeps the feature-selection decision (which features actually help) separate from feature engineering (creating candidate signal) — dimensionality reduction on the full engineered set will be handled explicitly in Notebook 06.

In [5]:
columns_to_drop = ["diag_1", "diag_2", "diag_3"]

for df_split in [train_df, val_df, test_df]:
    df_split.drop(columns=columns_to_drop, inplace=True)

engineered_feature_names = [
    "diag_1_category", "diag_2_category", "diag_3_category",
    "total_prior_utilization", "medications_per_day",
    "lab_procedures_per_day", "total_procedures",
    "num_medications_changed", "num_medications_prescribed",
]

print("Raw diagnosis columns dropped:", columns_to_drop)
print(f"\nNew engineered features added ({len(engineered_feature_names)}):")
for f in engineered_feature_names:
    print(" -", f)

print("\nUpdated shapes:")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Raw diagnosis columns dropped: ['diag_1', 'diag_2', 'diag_3']

New engineered features added (9):
 - diag_1_category
 - diag_2_category
 - diag_3_category
 - total_prior_utilization
 - medications_per_day
 - lab_procedures_per_day
 - total_procedures
 - num_medications_changed
 - num_medications_prescribed

Updated shapes:
Train: (63444, 53)
Validation: (15775, 53)
Test: (20124, 53)


## Feature and Target Separation

Identifiers (`encounter_id`, `patient_nbr`) and the target (`readmitted`) are separated from the predictive features, following the same convention as Notebook 03.

In [6]:
target_column = "readmitted"
identifier_columns = ["encounter_id", "patient_nbr"]

feature_columns = [
    col for col in train_df.columns
    if col not in identifier_columns + [target_column]
]

X_train = train_df[feature_columns].copy()
y_train = train_df[target_column].copy()

X_val = val_df[feature_columns].copy()
y_val = val_df[target_column].copy()

X_test = test_df[feature_columns].copy()
y_test = test_df[target_column].copy()

print("Feature matrix shapes:")
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print(f"\nNumber of features: {len(feature_columns)}")

Feature matrix shapes:
X_train: (63444, 50)
X_val: (15775, 50)
X_test: (20124, 50)

Number of features: 50


## Feature Types & Preprocessing Pipeline

Numerical features are median-imputed and standardized. Categorical features (including the three new diagnosis category columns) are most-frequent-imputed and one-hot encoded. The pipeline is fitted on the training data only.

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

numerical_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numerical features:", len(numerical_features))
print(numerical_features)
print("\nCategorical features:", len(categorical_features))
print(categorical_features)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numerical_features),
    ("categorical", categorical_pipeline, categorical_features)
])

preprocessor.fit(X_train)

X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("\nProcessed shapes:")
print("Train:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Test:", X_test_processed.shape)

Numerical features: 17
['admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'total_prior_utilization', 'medications_per_day', 'lab_procedures_per_day', 'total_procedures', 'num_medications_changed', 'num_medications_prescribed']

Categorical features: 33
['race', 'gender', 'age', 'payer_code', 'medical_specialty', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'diag_1_category', 'diag_2_category', 'diag_3_category']

Processed shapes:
Train: (63444,

## Retrain Champion Models on Engineered Features

The two champion models (CatBoost, LightGBM) plus the XGBoost reference model are retrained on the engineered feature set (229 features, down from the baseline's 2,304) and evaluated on the same validation set, using the same protocol as Notebook 04.

This directly tests whether the ~90% dimensionality reduction from clinical diagnosis grouping and the new derived features preserves — or improves — predictive performance relative to the raw one-hot baseline.

In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
import time

fe_models = {
    "CatBoost": CatBoostClassifier(random_state=RANDOM_STATE, verbose=0),
    "LightGBM": LGBMClassifier(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
    "XGBoost": XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss"),
}

def evaluate_model(name, model):
    start = time.time()
    model.fit(X_train_processed, y_train)
    train_time = time.time() - start

    y_pred = model.predict(X_val_processed)
    y_prob = model.predict_proba(X_val_processed)[:, 1]

    return {
        "Model": name,
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred),
        "Recall": recall_score(y_val, y_pred),
        "F1": f1_score(y_val, y_pred),
        "ROC-AUC": roc_auc_score(y_val, y_prob),
        "Train Time (s)": round(train_time, 2),
    }

fe_results_list = []
for name, model in fe_models.items():
    print(f"Training {name} on engineered features...")
    fe_results_list.append(evaluate_model(name, model))

fe_results_df = pd.DataFrame(fe_results_list).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)

print("\n" + "="*70)
print("FEATURE-ENGINEERED PERFORMANCE ON VALIDATION SET")
print("="*70)
print(fe_results_df.round(4).to_string(index=False))

Training CatBoost on engineered features...
Training LightGBM on engineered features...
Training XGBoost on engineered features...

FEATURE-ENGINEERED PERFORMANCE ON VALIDATION SET
   Model  Accuracy  Precision  Recall     F1  ROC-AUC  Train Time (s)
CatBoost    0.6375     0.6287  0.5491 0.5863   0.6870           19.46
LightGBM    0.6347     0.6255  0.5451 0.5825   0.6835            0.66
 XGBoost    0.6328     0.6203  0.5535 0.5850   0.6832            1.04


## Before / After Comparison & Log Results

The engineered feature set reduces dimensionality by ~90% (2,304 → 229 features) at the cost of a small ROC-AUC decrease (0.001–0.005 across models). This trade-off is logged transparently: the loss in raw discriminative power is minor, while the gains in training speed, interpretability, and reduced overfitting risk set up subsequent feature selection and optimization stages to be both faster and more robust.

Results are appended to the shared comparison log for the final end-to-end before/after report.

In [9]:
# Save feature-engineered results
fe_results_path = REPORTS_DIR / "feature_engineered_results.csv"
fe_results_df.to_csv(fe_results_path, index=False)

# Append to the running comparison log
comparison_log_path = REPORTS_DIR / "model_comparison_log.csv"
existing_log = pd.read_csv(comparison_log_path)

log_entry = fe_results_df.copy()
log_entry.insert(0, "stage", "02_feature_engineering")
log_entry.insert(1, "notebook", "05_feature_engineering")

updated_log = pd.concat([existing_log, log_entry], ignore_index=True)
updated_log.to_csv(comparison_log_path, index=False)

print(f"Feature-engineered results saved to: {fe_results_path}")
print(f"Comparison log updated: {comparison_log_path}")

# Direct before/after comparison table
comparison = updated_log[
    (updated_log["stage"].isin(["01_baseline", "02_feature_engineering"])) &
    (updated_log["Model"].isin(["CatBoost", "LightGBM", "XGBoost"]))
].pivot(index="Model", columns="stage", values="ROC-AUC")

comparison["Change"] = (
    comparison["02_feature_engineering"] - comparison["01_baseline"]
).round(4)

print("\n" + "="*60)
print("ROC-AUC: BASELINE vs FEATURE-ENGINEERED")
print("="*60)
print(comparison.round(4))

# Save fitted engineered-feature preprocessing and models for reuse
joblib.dump(preprocessor, MODELS_DIR / "fe_preprocessor.pkl")
fe_data = {
    "X_train": X_train_processed, "X_val": X_val_processed, "X_test": X_test_processed,
    "y_train": y_train, "y_val": y_val, "y_test": y_test,
}
joblib.dump(fe_data, MODELS_DIR / "fe_data.pkl")
joblib.dump(fe_models["CatBoost"], MODELS_DIR / "fe_catboost.pkl")
joblib.dump(fe_models["LightGBM"], MODELS_DIR / "fe_lightgbm.pkl")

print("\nEngineered preprocessing pipeline and data saved to models/")

Feature-engineered results saved to: e:\all projects\Machine Learning\Advanced-ML-Hospital-Readmission\reports\feature_engineered_results.csv
Comparison log updated: e:\all projects\Machine Learning\Advanced-ML-Hospital-Readmission\reports\model_comparison_log.csv

ROC-AUC: BASELINE vs FEATURE-ENGINEERED
stage     01_baseline  02_feature_engineering  Change
Model                                                
CatBoost       0.6918                  0.6870 -0.0048
LightGBM       0.6889                  0.6835 -0.0054
XGBoost        0.6845                  0.6832 -0.0013

Engineered preprocessing pipeline and data saved to models/


## Summary

This notebook engineered a compact, domain-informed feature set on top of the cleaned dataset, reusing the exact patient-level split from Notebook 03.

### Engineered Features

- **Clinical ICD-9 grouping**: `diag_1`, `diag_2`, `diag_3` (700+ raw codes each) were replaced with `diag_1_category`, `diag_2_category`, `diag_3_category` — 9 clinically meaningful groups (Circulatory, Respiratory, Digestive, Diabetes, Injury, Musculoskeletal, Genitourinary, Neoplasms, Other), following the grouping scheme from the original Strack et al. (2014) study.
- **Utilization features**: `total_prior_utilization`, `medications_per_day`, `lab_procedures_per_day`, `total_procedures`.
- **Medication summary features**: `num_medications_changed`, `num_medications_prescribed`.

### Impact

- Feature dimensionality after preprocessing dropped from **2,304 → 229** (≈90% reduction), driven almost entirely by replacing the raw diagnosis codes.
- Validation ROC-AUC decreased slightly across all three tracked models (CatBoost: -0.0048, LightGBM: -0.0054, XGBoost: -0.0013) — a minor, honestly-reported trade-off.
- Training time also decreased for CatBoost (28.0s → 19.5s) and LightGBM (0.82s → 0.66s), reflecting the much smaller feature space.

### Interpretation

The small ROC-AUC decrease suggests some fine-grained diagnostic signal was lost in grouping ICD-9 codes into 9 broad categories. However, the ~90% dimensionality reduction substantially improves interpretability, reduces overfitting risk on unseen data, and makes the next stages (feature selection, hyperparameter optimization) far more computationally tractable — a deliberate and documented trade-off rather than an unqualified improvement.

Results are logged in `reports/feature_engineered_results.csv` and appended to `reports/model_comparison_log.csv` under `stage = 02_feature_engineering`.

**Next notebook (06):** Feature selection on the 229-feature engineered set — testing whether removing low-signal features can recover some of this ROC-AUC gap while preserving (or further improving on) the dimensionality reduction achieved here.